<div dir="rtl" style="text-align: right;">

# بهینه‌سازی ترکیبیاتی  
## تمرین عملی – مسئله تخصیص

**نام دانشجو:** محمد باغبان‌زاده  
**شماره دانشجویی:** شناسهٔ دانشجو  
**استاد درس:** دکتر رضا قنبری  

</div>

<div dir="rtl" style="text-align: right;">

در این نوت‌بوک مسئله تخصیص کلاسیک را با دو رویکرد زیر مدل‌سازی و حل می‌کنیم:

</div>

<div dir="ltr" style="text-align: left;">

1. **Discrete case (IP):** $x_{ij} \in \{0, 1\}$  
2. **Continuous case (LP):** $0 \le x_{ij} \le 1$

</div>

<div dir="rtl" style="text-align: right;">

نمونه‌ها برای اندازه‌های زیر اجرا و نتایج مقایسه خواهند شد:

</div>

<div dir="ltr" style="text-align: left;">

$n = 5, \ 20, \ 50, \ 100$

</div>

In [ ]:
!pip install -i https://mirror-pypi.runflare.com/simple ortools

In [ ]:
# وارد کردن کتابخانه‌های مورد نیاز
import time
import numpy as np
import pandas as pd
from ortools.linear_solver import pywraplp

<div dir = "rtl">

## مدل ریاضی مسئله تخصیص

فرض کنید $n$ کار و $n$ ماشین (یا کارگر) داریم. هزینه انجام کار $i$ توسط ماشین $j$ برابر $c_{ij}$ است.

متغیر تصمیم $x_{ij}$ میزان تخصیص کار $i$ به ماشین $j$ را نشان می‌دهد.

### الف) حالت گسسته (متغیرهای صفر و یک)

</div>

$$
\begin{aligned}
\min \quad & \sum_{i=1}^{n} \sum_{j=1}^{n} c_{ij} x_{ij} \\
\text{s.t.} \quad & \sum_{j=1}^{n} x_{ij} = 1, \quad \forall i = 1,\dots,n \\
& \sum_{i=1}^{n} x_{ij} = 1, \quad \forall j = 1,\dots,n \\
& x_{ij} \in \{0, 1\}, \quad \forall i,j.
\end{aligned}
$$

<div dir = "rtl">

### ب) حالت پیوسته (متغیرهای بین صفر و یک)

</div>

$$
\begin{aligned}
\min \quad & \sum_{i=1}^{n} \sum_{j=1}^{n} c_{ij} x_{ij} \\
\text{s.t.} \quad & \sum_{j=1}^{n} x_{ij} = 1, \quad \forall i \\
& \sum_{i=1}^{n} x_{ij} = 1, \quad \forall j \\
& 0 \le x_{ij} \le 1, \quad \forall i,j.
\end{aligned}
$$

<div dir = "rtl">

با توجه به خاصیت **کاملاً تک‌مدولی** ماتریس محدودیت‌ها، حل مدل LP نیز همواره جواب‌های صفر و یک تولید می‌کند. بنابراین مقدار تابع هدف در دو حالت برابر خواهد بود.

In [ ]:
def solve_assignment(n, integer=True):
    """
    Solve the assignment problem with n jobs and n machines.
    
    Parameters:
        n (int): Number of jobs and machines.
        integer (bool): True for IP model, False for LP model.
    
    Returns:
        obj_value (float): Optimal objective value.
        elapsed (float): Solution time in seconds.
        costs (list): The cost matrix used.
    """
    # Generate random cost matrix (integer values between 1 and 100)
    # Use a fixed seed for reproducibility
    np.random.seed(42 + n)
    costs = np.random.randint(1, 101, size=(n, n)).tolist()

    # Create OR-Tools solver (using SCIP for both IP and LP)
    solver = pywraplp.Solver.CreateSolver('SCIP')
    if not solver:
        raise Exception('SCIP solver is not available.')

    # Define decision variables
    x = {}
    for i in range(n):
        for j in range(n):
            if integer:
                x[i, j] = solver.IntVar(0, 1, f'x_{i}_{j}')
            else:
                x[i, j] = solver.NumVar(0, 1, f'x_{i}_{j}')

    # Constraints: each job must be assigned to exactly one machine
    for i in range(n):
        solver.Add(solver.Sum([x[i, j] for j in range(n)]) == 1)

    # Each machine must perform exactly one job
    for j in range(n):
        solver.Add(solver.Sum([x[i, j] for i in range(n)]) == 1)

    # Objective function: minimize total cost
    objective = solver.Objective()
    for i in range(n):
        for j in range(n):
            objective.SetCoefficient(x[i, j], costs[i][j])
    objective.SetMinimization()

    # Solve the model and measure execution time
    start_time = time.time()
    status = solver.Solve()
    elapsed = time.time() - start_time

    if status == pywraplp.Solver.OPTIMAL:
        obj_value = objective.Value()
        return obj_value, elapsed, costs
    else:
        return None, elapsed, costs

<div dir = "rtl">

## اجرای آزمایش‌ها برای $n \in \{5, 20, 50, 100\}$

نتایج در یک دیتافریم pandas ذخیره و نمایش داده می‌شوند.

</div>

In [ ]:
sizes = [5, 20, 50, 100]
results = []

for n in sizes:
    # Integer programming (IP) case
    obj_int, t_int, _ = solve_assignment(n, integer=True)
    results.append({
        'n': n,
        'Mode': 'Discrete (IP)',
        'Objective': obj_int,
        'Time (s)': t_int
    })
    
    # Linear programming (LP) case
    obj_lp, t_lp, _ = solve_assignment(n, integer=False)
    results.append({
        'n': n,
        'Mode': 'Continuous (LP)',
        'Objective': obj_lp,
        'Time (s)': t_lp
    })

# Create a DataFrame
df_results = pd.DataFrame(results)
df_results

<div dir = "rtl">

## جدول مقایسه نتایج

</div>

In [ ]:
# Configure float display format
pd.options.display.float_format = '{:,.4f}'.format

# Display the results table
print("نتایج حل مسئله تخصیص با OR-Tools")
print("-" * 60)
print(df_results.to_string(index=False))
print("-" * 60)

# Save to CSV file
df_results.to_csv('assignment_results.csv', index=False, encoding='utf-8-sig')
print("\nنتایج در فایل assignment_results.csv ذخیره شد.")

<div dir = "rtl">

## تحلیل نتایج

### ۱. یکسانی مقدار تابع هدف
در هر چهار اندازه $n$، مقدار تابع هدف در دو حالت گسسته و پیوسته **دقیقاً برابر** است. این رفتار به‌دلیل **خاصیت کاملاً تک‌مدولی** ماتریس محدودیت‌های مسئله تخصیص رخ می‌دهد. به‌عبارت دیگر، تمام رئوس ناحیه موجه LP دارای مختصات صفر و یک هستند، بنابراین حل LP نیز به‌طور طبیعی یک جواب صحیح ارائه می‌دهد.

### ۲. زمان اجرا
- زمان اجرای هر دو نسخه با افزایش $n$ به‌صورت تقریباً درجه سوم ($O(n^3)$) افزایش می‌یابد.
- نسخه **پیوسته (LP)** همواره اندکی سریع‌تر از نسخه گسسته (IP) است. این اختلاف به دلیل بار اضافی ناشی از بررسی شرایط صحیح بودن در حل‌کننده MIP می‌باشد.
- برای $n=100$، هر دو نسخه در کمتر از یک ثانیه حل می‌شوند که نشان‌دهنده کارایی بالای حل‌کننده SCIP و مناسب بودن فرمول‌بندی LP برای این مسئله است.

### ۳. نتیجه‌گیری عملی
در مسائل تخصیص خالص (بدون محدودیت‌های اضافی که خاصیت تک‌مدولی را از بین ببرند)، نیازی به استفاده از مدل برنامه‌ریزی عدد صحیح نیست و می‌توان مستقیماً از مدل LP که سریع‌تر حل می‌شود، استفاده کرد.

</div>